# Predikcija zadovoljstva putnika avio-kompanije primenom neuronskih mreža i algoritama mašinskog učenja

Koristiće se:
- Logistička regresija
- Random Forest
- Višeslojna neuronska mreža (MLP)

# Biblioteke

In [101]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier

In [102]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# Priprema podataka

In [103]:
# Nepotrebne kolone uklanjamo
train_df.drop(columns=['id', 'Unnamed: 0'], inplace=True)
test_df.drop(columns=['id', 'Unnamed: 0'], inplace=True)

In [ ]:
# Na osnovu zadnjih analiza, vidimo da su neke od kolona koje predstavljaju usluge avio kompanije imale vrednost 0, sto nije primenljivo. 
# Ove vrednosti cemo zameniti sa NaN, a zatim popuniti sa medianom.
service_cols = [
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Inflight service',
    'Cleanliness'
]

train_df[service_cols] = train_df[service_cols].replace(0, np.nan)
test_df[service_cols] = test_df[service_cols].replace(0, np.nan)
median = train_df[service_cols].median()

train_df[service_cols] = train_df[service_cols].fillna(median)
test_df[service_cols] = test_df[service_cols].fillna(median)

In [105]:
# Na osnovu korelacione matrice iz EDA, brišemo Arrival Delay in Minutes, jer je visoko korelisana sa Departure Delay in Minutes
train_df.drop(columns=['Arrival Delay in Minutes'], inplace=True)
test_df.drop(columns=['Arrival Delay in Minutes'], inplace=True)

In [106]:
# Kodiranje kategorickih kolona-One Hot Encoding
categorical_cols = [
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class'
]

train_df = pd.get_dummies(train_df, columns=categorical_cols, dtype=int , drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, dtype=int , drop_first=True)

In [107]:
# Kako bi se osigurali da train i test skup imaju iste kolone nakon one hot encoding-a, koristimo align funkciju
train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)


In [113]:
print("Kolone se poklapaju:", test_df.columns.equals(train_df.columns))

Kolone se poklapaju: True


In [109]:
le = LabelEncoder()

train_df['satisfaction'] = le.fit_transform(train_df['satisfaction'])
test_df['satisfaction'] = le.transform(test_df['satisfaction'])

In [110]:
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

{'neutral or dissatisfied': np.int64(0), 'satisfied': np.int64(1)}


In [111]:
# Train test split
X_train = train_df.drop('satisfaction', axis=1)
y_train = train_df['satisfaction']

X_test = test_df.drop('satisfaction', axis=1)
y_test = test_df['satisfaction']